# E11_Research — PPO со слоем понятий FCA

**Модуль курса:** [/fca-rl](https://rl-cuber-unity-code.com/fca-rl) —
«FCA + RL: формальный анализ концептов»
**Что реализуется:** формальный анализ понятий по бинарным признакам
состояния и слой принадлежности понятиям **внутри графа политики**.
**Ожидаемый результат:** **≥ 70 %** пройденных эпизодов; главное же —
сравнение числа шагов до порога с обычной MLP-политикой.
**Время обучения:** около 40 минут на сид (`QUICK_RUN = True` — около
четырёх минут).

## Это исследовательский слот

Здесь проверяется **гипотеза**, а не воспроизводится известный метод.
Отрицательный результат так же ценен, как положительный.

### Гипотеза

Задача иерархична: «взять ключ» → «дойти до двери» → «дойти до цели».
Обычная MLP-политика вынуждена вывести эту структуру из наград сама
и тратит на это часть бюджета.

Формальный анализ понятий (FCA) находит в бинарной таблице
«состояние — признак» устойчивые сочетания. Поданные политике как
дополнительные входы, они дают готовые составные признаки вида
«есть ключ **и** я справа от стены» — то есть именно те подцели.

### Что измеряется

Число шагов среды до порога у `fca_ppo` против `ppo_discrete` на той же
среде и тех же гиперпараметрах. Конфиги отличаются **только** блоком `fca`.

**Сравнивать надо шаги до порога, а не итоговую долю успехов**: смоук-прогон
показал, что FCA-политика доходит до 100 % уже на 30 000 шагов, то есть
итоговое значение упрётся в потолок у обоих методов.

### Встроенная проверка работоспособности метода

Признаки «есть ключ» и «дверь открыта» в этой среде **эквивалентны
по построению**. FCA обязан обнаружить это сам и склеить их в одно понятие.
Если в списке понятий они встречаются по отдельности — метод не работает,
и дальше смотреть незачем.

## Требования

```powershell
python scripts\build_env.py E11_Research
```

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

## 2. Конфигурация

Значения совпадают с `configs/E11_Research__fca_ppo.yaml`.

Про `min_support`. Сочетание признаков, встретившееся в трёх состояниях
из тысяч, — шум опыта, а не структура задачи. Без отсечения число понятий
растёт, а польза нет.

Про `max_intent`. Понятие из шести признаков почти наверняка описывает
частный случай, а не подцель, и добавляет политике вход, истинный
в единицах состояний.

In [ ]:
QUICK_RUN = True   # True — быстрая проверка (~4 мин); False — полное обучение (~40 мин)

from labrl.algos.ppo import PPOConfig
from labrl.eval.success import TERMINATED
from labrl.train.onpolicy import OnPolicyTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 30_000 if QUICK_RUN else 300_000

# --- параметры решётки понятий ---
NUM_ATTRIBUTES = 7        # столько первых признаков наблюдения бинарны
WARMUP_STEPS = 2_000      # шагов случайной политикой до построения решётки
MIN_SUPPORT = 0.02        # минимальная доля состояний в объёме понятия
MAX_INTENT = 4            # максимум признаков в содержании
SHARPNESS = 20.0          # крутизна сигмоид бинаризации и принадлежности

ALGO_CFG = PPOConfig(
    gamma=0.99, gae_lambda=0.95, learning_rate=3e-4, clip_range=0.2,
    epochs=4, minibatch_size=512, value_coef=0.5, entropy_coef=0.01,
    max_grad_norm=0.5, normalize_advantage=True, target_kl=0.0,
)

TRAIN_CFG = OnPolicyTrainConfig(
    total_steps=TOTAL_STEPS,
    rollout_steps=256,
    eval_every_steps=max(1, TOTAL_STEPS // 10),
    eval_episodes=30,
    success_rule=TERMINATED,     # успех = цель достигнута, а не «награда выше порога»
)

# Расписание НЕ сжимается вместе с сокращённым прогоном (T-10).
LEARNING_RATE = LinearSchedule(start=3e-4, end=5e-5, decay_steps=300_000)

print(f"бюджет: {TOTAL_STEPS} шагов, прогрев для решётки: {WARMUP_STEPS}")

## 3. Подключение среды

In [ ]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E11_Research"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED,
                        time_scale=20.0, no_graphics=True, num_areas=8)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен: {vec.num_envs}")

## 4. Инспекция пространств

Проверка против строки-контракта `ENV_SPEC.md` — и, отдельно, проверка того,
что **первые семь признаков действительно бинарны**. Формальный контекст
из непрерывных чисел построить нельзя: `FormalContext` такой вход отвергает.

In [ ]:
OBS_DIM = vec.single_obs_dim
BRANCHES = tuple(vec.action_spec.discrete_branches)

print(f"obs_dim: {OBS_DIM}, ветки: {BRANCHES}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=16; discrete_branches=4; continuous_size=0; max_step=200 -->
assert [tuple(s) for s in vec.obs_shapes] == [(16,)], vec.obs_shapes
assert OBS_DIM == 16 and BRANCHES == (4,)
assert vec.action_spec.continuous_size == 0

flat = vec.flatten_obs(obs)
binary_part = flat[:, :NUM_ATTRIBUTES]
assert np.isin(binary_part, (0.0, 1.0)).all(), (
    "первые признаки обязаны быть бинарными: формальный контекст из непрерывных "
    "чисел построить нельзя")

print(f"\nбинарная часть наблюдения (арена 0): {binary_part[0].astype(int).tolist()}")
print("размерности совпадают с ENV_SPEC.md, признаки 0–6 бинарны")

## 5. Формальный контекст и решётка понятий

Опыт собирается **случайной** политикой. Он должен быть разнообразным,
а не хорошим: понятия описывают структуру пространства состояний,
а не удачные траектории.

In [ ]:
from labrl.nets.fca import KEYDOOR_ATTRIBUTES, build_concept_layer

rng = np.random.default_rng(SEED)
collected = []
obs_now = vec.reset()
for _ in range(WARMUP_STEPS):
    collected.append(vec.flatten_obs(obs_now).copy())
    actions = np.stack([rng.integers(0, b, size=vec.num_envs) for b in BRANCHES], axis=1)
    obs_now = vec.step(actions.astype(np.int32)).obs
experience = np.concatenate(collected, axis=0)

concept_layer, context = build_concept_layer(
    experience, num_attributes=NUM_ATTRIBUTES,
    min_support=MIN_SUPPORT, max_intent=MAX_INTENT,
    attribute_names=list(KEYDOOR_ATTRIBUTES),
)
concept_layer.binarize_sharpness = SHARPNESS
concept_layer.concept_sharpness = SHARPNESS

print(context.describe())
print()
print(concept_layer.describe(list(KEYDOOR_ATTRIBUTES)))

### Проверка работоспособности метода

Признаки «есть ключ» и «дверь открыта» эквивалентны по построению среды.
Если FCA не склеил их в одно понятие — метод не работает, и результаты
обучения ничего не скажут.

In [ ]:
from labrl.utils.fca import implications

KEY, DOOR = 0, 1
together = sum(1 for c in concept_layer.concepts if KEY in c.intent and DOOR in c.intent)
apart = sum(1 for c in concept_layer.concepts
            if (KEY in c.intent) != (DOOR in c.intent))

print(f"понятий, где «есть ключ» и «дверь открыта» вместе: {together}")
print(f"понятий, где они порознь:                          {apart}")
assert apart == 0, (
    "FCA не обнаружил эквивалентность двух признаков — метод не работает; "
    "проверьте бинаризацию и min_support")
print("\nэквивалентность обнаружена: метод работает как задумано")

found = implications(context, min_support=MIN_SUPPORT)
print(f"\nвсего импликаций «набор ⇒ признак»: {len(found)}")
for premise, consequence in found[:6]:
    left = ", ".join(KEYDOOR_ATTRIBUTES[i] for i in premise)
    print(f"  {{{left}}} ⇒ {KEYDOOR_ATTRIBUTES[consequence]}")

## 6. Нейросеть со слоем понятий

Наблюдение подаётся **целиком**, признаки понятий добавляются к нему
конкатенацией. Это методологически важно: понятия **дополняют** признаки,
а не заменяют их. Замена была бы сильным утверждением — что вся нужная
информация выражается понятиями, — а проверяется более скромное:
помогают ли они.

In [ ]:
from labrl.nets.fca import FCAPolicy
from labrl.nets.mlp import MLPValueNetwork

policy_net = FCAPolicy(OBS_DIM, BRANCHES, concept_layer,
                       hidden_sizes=(128, 128), activation="relu")
value_net = MLPValueNetwork(OBS_DIM, (128, 128), "relu")

with torch.no_grad():
    probe = torch.zeros(3, OBS_DIM)
    assert policy_net(probe).shape == (3, sum(BRANCHES))
    assert value_net(probe).shape == (3, 1)

print(f"понятий на входе MLP: {concept_layer.num_concepts}")
print(f"вход MLP: {OBS_DIM} + {concept_layer.num_concepts} = "
      f"{OBS_DIM + concept_layer.num_concepts}")
print(f"параметров актора:  {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров критика: {sum(p.numel() for p in value_net.parameters()):,}")
print(f"\nобучаемых параметров в слое понятий: "
      f"{sum(p.numel() for p in concept_layer.parameters())} "
      "(веса найдены данными и обучению не подлежат)")

## 7. Теория: почему слой понятий экспортируется в ONNX

Требование 10.7 прямое: всё, что преобразует наблюдение, обязано попасть
в граф. Признаки, посчитанные в Python и не уехавшие в Unity, —
гарантированное расхождение обучения и инференса, причём **молчаливое**:
сеть в Unity получит другой вход и будет работать, просто плохо.

Принадлежность понятию выражена так, чтобы её принимал `torch.onnx.export`
при opset 9:

$$a = \sigma\big(k\,(x - \theta)\big), \qquad
s_k = a \cdot \text{intent}_k^{\top} - \big(|\text{intent}_k| - \tfrac12\big),
\qquad m_k = \sigma(\beta\, s_k)$$

Обе строки — `Linear` с **фиксированными** весами плюс сигмоида, то есть
`MatMul`, `Add`, `Sigmoid`.

**Смысл смещения** $|\text{intent}| - \frac12$. Скалярное произведение
бинарного вектора признаков и бинарного содержания равно числу совпавших
признаков. Оно равно $|\text{intent}|$ тогда и только тогда, когда есть все;
на единицу меньше — если не хватает хотя бы одного. Порог ровно посередине
разделяет эти случаи максимально далеко от обоих.

**Почему мягкая проверка.** Жёсткое «все признаки на месте» — это $\min$
по признакам, и производной у него почти нигде нет. При $\beta = 20$ мягкая
версия численно неотличима от жёсткой на целых входах, но дифференцируема.

Ниже — прямой проход слоя, тот самый, что уедет в граф.

In [ ]:
import inspect

from labrl.nets.fca import FCAConceptLayer

print(inspect.getsource(FCAConceptLayer.forward))

# Мягкая проверка обязана совпадать с жёсткой на целых входах.
sample = experience[:256, :NUM_ATTRIBUTES]
with torch.no_grad():
    soft = concept_layer(torch.as_tensor(sample, dtype=torch.float32)).numpy()
exact = concept_layer.membership_exact(sample)
print(f"max|мягкая − жёсткая| на реальных состояниях: {np.abs(soft - exact).max():.2e}")
assert np.abs(soft - exact).max() < 1e-3

## 8. Цикл обучения

In [ ]:
from labrl.algos.ppo_discrete import PPODiscrete
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.onpolicy import train_on_policy

algo = PPODiscrete(policy_net, value_net, ALGO_CFG, device=DEVICE, seed=SEED)
run = create_run_dir(ENV_ID, "fca_ppo", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN,
                    "num_concepts": concept_layer.num_concepts})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_on_policy(
    vec=vec, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    lr_schedule=LEARNING_RATE,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.3f}, дошло {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={
    "seed": SEED, "num_concepts": concept_layer.num_concepts,
    "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, обновлений: {result.updates}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

## 9. Оценка и сравнение

Главная диагностика среды — **`Env/KeyRate`**: пока она около нуля,
обсуждать доли успехов бессмысленно, агент не прошёл первый этап.

Контрольный прогон без слоя понятий запускается отдельно:

```powershell
python scripts\train.py --config configs\E11_Research__ppo_discrete.yaml --all-seeds
```

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

steps = [s for s, _, _ in result.eval_history]
success = [v for _, _, v in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]

axes[0].plot(steps, success, marker="o", color="tab:green")
axes[0].axhline(0.7, color="k", ls="--", lw=1, label="критерий приёмки")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Eval/Success Rate")
axes[0].set_xlabel("шаг сбора"); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(steps, rewards, marker="o", color="tab:orange")
axes[1].set_title("Eval/Mean Reward")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Шаги до порога — то, что сравнивается с контрольным прогоном.
THRESHOLD = 0.70
reached = next((s for s, _, v in result.eval_history if v >= THRESHOLD), None)
print(f"понятий в политике: {concept_layer.num_concepts}")
print(f"итоговая доля успехов: {result.last_eval_success:.0%}")
print(f"шагов до порога {THRESHOLD:.0%}: {reached if reached is not None else 'не достигнут'}")
print("\nсравнивайте именно эту величину с контрольным прогоном без слоя понятий:")
print("  python scripts\\train.py --config configs\\E11_Research__ppo_discrete.yaml --seed 0")

if QUICK_RUN:
    print(f"\nбыстрый прогон: критерий приёмки ({THRESHOLD:.0%}) не проверяется")
else:
    assert result.last_eval_success >= THRESHOLD, (
        f"не достигнут критерий приёмки: {result.last_eval_success:.0%}")
    print(f"\nкритерий приёмки достигнут: {result.last_eval_success:.0%}")

## 10. Экспорт в ONNX и возврат в Unity

Слой понятий уезжает в граф **вместе с политикой**. Проверка числового
паритета здесь особенно важна: если бы слой остался в Python, ONNX всё равно
прошёл бы проверку структуры — и разошёлся бы с обучением на реальных данных.

In [ ]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из реального распределения среды (требование 10.5), а не шум.
collected = []
obs_now = vec.reset()
while sum(len(c) for c in collected) < 64:
    collected.append(obs_now[0].copy())
    obs_now = vec.step(algo.deterministic_action(vec.flatten_obs(obs_now))).obs
sample_obs = [np.concatenate(collected, axis=0)[:64]]

spec = ActionSpecLite(discrete_branches=BRANCHES, continuous_size=0)
policy_module = algo.policy_module()
onnx_path = export_policy_to_onnx(policy_module, spec, vec.obs_shapes,
                                  run.onnx / "policy.onnx", strategy="categorical")

verification = verify_onnx_model(onnx_path, policy_module, spec, vec.obs_shapes,
                                 sample_obs=sample_obs)
print(verification.report())
verification.raise_if_failed()

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
graph_action = session.run(
    ["deterministic_discrete_actions"],
    {"obs_0": sample_obs[0],
     "action_masks": np.ones((64, sum(BRANCHES)), dtype=np.float32),
     "recurrent_in": np.zeros((64, 1, 0), dtype=np.float32)},
)[0]
python_action = algo.deterministic_action(sample_obs[0])
mismatch = int((graph_action != python_action).sum())
print(f"\nрасхождений действия ONNX и Python: {mismatch} из {python_action.size}")
assert mismatch == 0, "слой понятий не попал в граф — инференс в Unity будет другим"

if not QUICK_RUN:
    # Модель проекта обновляется только полным прогоном (T-14).
    model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                  "Models" / f"{ENV_ID}_fca_ppo.onnx")
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    model_dest.write_bytes(onnx_path.read_bytes())
    print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
else:
    print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

vec.handle.close()

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E11_Research/Scenes/E11_Research.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента `Agent`: `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: агент идёт к жёлтому ключу, дверь исчезает, агент идёт к зелёной цели.

### Что дальше

Слот исследовательский, и продолжений у него несколько:

| Направление | Что менять |
|---|---|
| несколько ключей и дверей | `KeyDoorArea`, признаки контекста |
| понятия, перестраиваемые по этапам | пересоздавать `FCAPolicy` между этапами (TS11-3) |
| FCA-анализ гиперпараметров | объекты контекста — конфигурации прогонов, признаки — «сходится за < 500k шагов» и т. п. (модуль `/fca-rl`, первый раздел) |

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, стрелки или **WASD**.